In [3]:
"""
WBPR-LightGCN 训练脚本
========================================
数据集  : Amazon_Reviews_Processed
核心改动: build_normalized_adjacency 中边权 w = rating / 5，其余架构与原版 LightGCN 完全一致
训练方式: 全量数据训练（无 train/test 划分），loss 收敛即停止
评分归一化: w = r / 5  →  权重范围 [0.2, 1.0]
========================================
运行方式:
    python train_wbpr_lightgcn.py
依赖: torch, numpy, pandas, scipy
"""

import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import scipy.sparse as sp
from collections import defaultdict

# ─────────────────────────────────────────────────────────────
# 路径与超参数（直接在这里修改）
# ─────────────────────────────────────────────────────────────
DATA_PATH        = "/Users/yubinhao/ml-1m/Amazon Review/数据集/Amazon_Reviews_Processed"
SAVE_DIR         = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/模型文件"

MIN_INTERACTIONS = 5       # 用户最少交互数（过滤冷启动用户）
EMBED_DIM        = 64      # embedding 维度
N_LAYERS         = 3       # LightGCN 传播层数
LR               = 1e-3    # 学习率
WEIGHT_DECAY     = 1e-4    # L2 正则
BATCH_SIZE       = 2048    # BPR 采样批大小
EPOCHS           = 100     # 训练轮数
LOG_EVERY        = 10      # 每隔几个 epoch 打印一次 loss
RATING_MAX       = 5.0     # 权重归一化分母：w = rating / RATING_MAX
SEED             = 42


# ─────────────────────────────────────────────────────────────
# 工具
# ─────────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


# ─────────────────────────────────────────────────────────────
# 数据加载与预处理
# ─────────────────────────────────────────────────────────────
def load_and_preprocess(data_path, min_interactions=5):
    print("[数据] 读取数据集...")
    df = pd.read_csv(data_path, usecols=["user_id", "item_id", "rating"])
    print(f"  原始记录数: {len(df)}, 用户数: {df['user_id'].nunique()}, 物品数: {df['item_id'].nunique()}")

    # 过滤冷启动用户
    user_counts = df.groupby("user_id").size()
    valid_users = user_counts[user_counts >= min_interactions].index
    df = df[df["user_id"].isin(valid_users)].copy()
    print(f"  过滤后记录数: {len(df)}, 用户数: {df['user_id'].nunique()}, 物品数: {df['item_id'].nunique()}")

    # 字符串 ID → 连续整数
    user_ids = sorted(df["user_id"].unique())
    item_ids = sorted(df["item_id"].unique())
    user2idx = {u: i for i, u in enumerate(user_ids)}
    item2idx = {it: i for i, it in enumerate(item_ids)}
    idx2user = {i: u for u, i in user2idx.items()}
    idx2item = {i: it for it, i in item2idx.items()}

    df["uid"] = df["user_id"].map(user2idx)
    df["iid"] = df["item_id"].map(item2idx)

    n_users = len(user_ids)
    n_items = len(item_ids)
    print(f"  编码后: {n_users} 用户, {n_items} 物品")

    return df, n_users, n_items, user2idx, item2idx, idx2user, idx2item


# ─────────────────────────────────────────────────────────────
# 构造加权归一化邻接矩阵（WBPR 核心改动）
# ─────────────────────────────────────────────────────────────
def build_normalized_adjacency(df, n_users, n_items, rating_max=5.0):
    """
    原始 LightGCN : A[u, i] = 1
    WBPR-LightGCN: A[u, i] = rating / rating_max  ∈ [0.2, 1.0]

    归一化公式（与 LightGCN 原文一致，度替换为加权度）：
        hat_A[u,i] = w_ui / sqrt(d_u_w * d_i_w)
        d_u_w = sum_i(w_ui)
    """
    print("[邻接] 构造加权归一化邻接矩阵...")

    uids    = df["uid"].values
    iids    = df["iid"].values
    weights = (df["rating"].values / rating_max).astype(np.float32)

    N = n_users + n_items

    # 对称边（user→item 和 item→user 权重相同）
    all_row = np.concatenate([uids,           iids + n_users])
    all_col = np.concatenate([iids + n_users, uids          ])
    all_w   = np.concatenate([weights,        weights       ])

    A = sp.csr_matrix((all_w, (all_row, all_col)), shape=(N, N), dtype=np.float32)

    # 加权度归一化
    d_w          = np.array(A.sum(axis=1)).flatten()
    d_w_inv_sqrt = np.where(d_w > 0, 1.0 / np.sqrt(d_w), 0.0)
    D_inv_sqrt   = sp.diags(d_w_inv_sqrt)
    hat_A        = D_inv_sqrt @ A @ D_inv_sqrt

    hat_A_coo = hat_A.tocoo()
    indices   = torch.LongTensor(np.vstack([hat_A_coo.row, hat_A_coo.col]))
    values    = torch.FloatTensor(hat_A_coo.data)
    hat_A_torch = torch.sparse_coo_tensor(indices, values, (N, N))

    print(f"  邻接矩阵: {N}×{N}, 非零元素: {hat_A_coo.nnz}")
    return hat_A_torch


# ─────────────────────────────────────────────────────────────
# WBPR-LightGCN 模型
# ─────────────────────────────────────────────────────────────
class WBPRLightGCN(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, n_layers, hat_A):
        super().__init__()
        self.n_users   = n_users
        self.n_items   = n_items
        self.embed_dim = embed_dim
        self.n_layers  = n_layers

        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

        # hat_A 不用 register_buffer，固定留在 CPU（MPS 不支持稀疏张量）
        self.hat_A = hat_A

    def forward(self):
        """逐层传播后各层平均，返回最终 user/item embedding。
        稀疏乘法在 CPU 完成（MPS 不支持稀疏算子），结果搬回原设备。
        """
        dev = self.user_emb.weight.device
        E0  = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)

        all_layers = [E0]
        E = E0
        for _ in range(self.n_layers):
            # hat_A 固定在 CPU；E 临时搬到 CPU 做稀疏乘法后再搬回 dev
            E = torch.sparse.mm(self.hat_A, E.cpu()).to(dev)
            all_layers.append(E)

        E_final    = torch.stack(all_layers, dim=1).mean(dim=1)
        user_final = E_final[:self.n_users]
        item_final = E_final[self.n_users:]
        return user_final, item_final

    def bpr_loss(self, user_final, item_final, uids, pos_iids, neg_iids):
        u_emb   = user_final[uids]
        pos_emb = item_final[pos_iids]
        neg_emb = item_final[neg_iids]

        pos_score = (u_emb * pos_emb).sum(dim=1)
        neg_score = (u_emb * neg_emb).sum(dim=1)
        loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

        reg_loss = (
            self.user_emb.weight[uids].norm(2).pow(2) +
            self.item_emb.weight[pos_iids].norm(2).pow(2) +
            self.item_emb.weight[neg_iids].norm(2).pow(2)
        ) / len(uids)

        return loss + WEIGHT_DECAY * reg_loss


# ─────────────────────────────────────────────────────────────
# BPR 采样
# ─────────────────────────────────────────────────────────────
def build_user_pos_dict(df):
    user_pos = defaultdict(set)
    for row in df.itertuples(index=False):
        user_pos[row.uid].add(row.iid)
    return user_pos


def sample_bpr_batch(all_uids, all_iids, user_pos_dict, n_items, batch_size):
    idx      = np.random.choice(len(all_uids), size=batch_size, replace=True)
    uids     = all_uids[idx]
    pos_iids = all_iids[idx]

    neg_iids = np.random.randint(0, n_items, size=batch_size)
    for j in range(batch_size):
        while neg_iids[j] in user_pos_dict[uids[j]]:
            neg_iids[j] = np.random.randint(0, n_items)

    return uids, pos_iids, neg_iids


# ─────────────────────────────────────────────────────────────
# 保存
# ─────────────────────────────────────────────────────────────
def save_checkpoint(model, tag, epoch, loss,
                    user2idx, item2idx, idx2user, idx2item,
                    n_users, n_items):
    """
    保存内容：
      model_{tag}.pt       — 完整 state_dict（用于继续训练或恢复）
      embeddings_{tag}.pt  — 最终 user/item embedding（直接输入 pattern 框架）
      id_mappings.pt       — user2idx / item2idx / idx2user / idx2item
      config.pt            — 超参数与数据统计
    """
    # 1. 模型权重
    torch.save(
        {"epoch": epoch, "loss": loss, "model_state_dict": model.state_dict()},
        os.path.join(SAVE_DIR, f"model_{tag}.pt")
    )

    # 2. 最终 embedding（detach → CPU）
    model.eval()
    with torch.no_grad():
        user_final, item_final = model()
    torch.save(
        {"user_embeddings": user_final.cpu(),
         "item_embeddings": item_final.cpu(),
         "epoch": epoch, "loss": loss},
        os.path.join(SAVE_DIR, f"embeddings_{tag}.pt")
    )

    # 3. ID 映射（只写一次）
    mapping_path = os.path.join(SAVE_DIR, "id_mappings.pt")
    if not os.path.exists(mapping_path):
        torch.save(
            {"user2idx": user2idx, "item2idx": item2idx,
             "idx2user": idx2user, "idx2item": idx2item},
            mapping_path
        )

    # 4. 配置（只写一次）
    config_path = os.path.join(SAVE_DIR, "config.pt")
    if not os.path.exists(config_path):
        torch.save(
            {"n_users": n_users, "n_items": n_items,
             "embed_dim": EMBED_DIM, "n_layers": N_LAYERS,
             "rating_max": RATING_MAX,
             "min_interactions": MIN_INTERACTIONS,
             "seed": SEED},
            config_path
        )

    print(f"    → 已保存 [{tag}]  epoch={epoch}, loss={loss:.4f}")


# ─────────────────────────────────────────────────────────────
# 主流程
# ─────────────────────────────────────────────────────────────
def main():
    set_seed(SEED)
    device = get_device()
    print(f"[设备] 使用: {device}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    # 1. 全量数据加载
    df, n_users, n_items, user2idx, item2idx, idx2user, idx2item = \
        load_and_preprocess(DATA_PATH, MIN_INTERACTIONS)

    # 2. 邻接矩阵（基于全量数据）
    # 注意：MPS 不支持稀疏张量，hat_A 固定留在 CPU，forward() 内稀疏乘法在 CPU 完成
    hat_A = build_normalized_adjacency(df, n_users, n_items, RATING_MAX)

    # 3. 模型与优化器
    model     = WBPRLightGCN(n_users, n_items, EMBED_DIM, N_LAYERS, hat_A).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # 4. 采样准备
    all_uids      = df["uid"].values
    all_iids      = df["iid"].values
    user_pos_dict = build_user_pos_dict(df)
    n_batches     = max(1, len(all_uids) // BATCH_SIZE)

    print(f"\n[训练] 全量数据训练，共 {EPOCHS} epoch，每 epoch {n_batches} batch")
    print(f"  embed_dim={EMBED_DIM}, n_layers={N_LAYERS}, lr={LR}, "
          f"weight_decay={WEIGHT_DECAY}, batch_size={BATCH_SIZE}\n")

    best_loss  = float("inf")
    best_epoch = 0
    avg_loss   = 0.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()

        for _ in range(n_batches):
            uids_b, pos_b, neg_b = sample_bpr_batch(
                all_uids, all_iids, user_pos_dict, n_items, BATCH_SIZE
            )
            uids_t = torch.LongTensor(uids_b).to(device)
            pos_t  = torch.LongTensor(pos_b).to(device)
            neg_t  = torch.LongTensor(neg_b).to(device)

            optimizer.zero_grad()
            user_final, item_final = model()
            loss = model.bpr_loss(user_final, item_final, uids_t, pos_t, neg_t)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / n_batches
        elapsed  = time.time() - t0

        if epoch % LOG_EVERY == 0 or epoch == 1:
            print(f"  Epoch {epoch:>3d}/{EPOCHS}  loss={avg_loss:.4f}  [{elapsed:.1f}s]")

        # 保存 loss 最低的 checkpoint
        if avg_loss < best_loss:
            best_loss  = avg_loss
            best_epoch = epoch
            save_checkpoint(model, "best", epoch, avg_loss,
                            user2idx, item2idx, idx2user, idx2item,
                            n_users, n_items)

    # 保存最终 epoch 的 checkpoint
    save_checkpoint(model, "final", EPOCHS, avg_loss,
                    user2idx, item2idx, idx2user, idx2item,
                    n_users, n_items)

    print(f"\n[完成] 训练结束")
    print(f"  最低 loss={best_loss:.4f}（Epoch {best_epoch}）")
    print(f"  模型文件保存至: {SAVE_DIR}")


if __name__ == "__main__":
    main()

[设备] 使用: mps
[数据] 读取数据集...
  原始记录数: 491708, 用户数: 94329, 物品数: 173847
  过滤后记录数: 363218, 用户数: 27794, 物品数: 144853
  编码后: 27794 用户, 144853 物品
[邻接] 构造加权归一化邻接矩阵...
  邻接矩阵: 172647×172647, 非零元素: 722798

[训练] 全量数据训练，共 100 epoch，每 epoch 177 batch
  embed_dim=64, n_layers=3, lr=0.001, weight_decay=0.0001, batch_size=2048

  Epoch   1/100  loss=0.6714  [69.2s]
    → 已保存 [best]  epoch=1, loss=0.6714
    → 已保存 [best]  epoch=2, loss=0.6015
    → 已保存 [best]  epoch=3, loss=0.4656
    → 已保存 [best]  epoch=4, loss=0.3729
    → 已保存 [best]  epoch=5, loss=0.3161
    → 已保存 [best]  epoch=6, loss=0.2751
    → 已保存 [best]  epoch=7, loss=0.2409
    → 已保存 [best]  epoch=8, loss=0.2145
    → 已保存 [best]  epoch=9, loss=0.1902
  Epoch  10/100  loss=0.1688  [64.5s]
    → 已保存 [best]  epoch=10, loss=0.1688
    → 已保存 [best]  epoch=11, loss=0.1515
    → 已保存 [best]  epoch=12, loss=0.1352
    → 已保存 [best]  epoch=13, loss=0.1226
    → 已保存 [best]  epoch=14, loss=0.1103
    → 已保存 [best]  epoch=15, loss=0.1004
    → 已保存 [best]  epo